In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.linear_model import Perceptron as SklearnPerceptron


np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)
DATA_URL = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
            "00267/data_banknote_authentication.txt")

LOCAL_FILE = "data_banknote_authentication.csv"
COLUMNS = ["Variance", "Skewness", "Curtosis", "Entropy", "Class"]
def dataset():
    try:
        urllib.request.urlretrieve(DATA_URL, LOCAL_FILE)
        print("Dataset downloaded from UCI repository.")
    except Exception as e:
        print(f"Could not download dataset automatically ({e}). "

              f"Looking for a local copy: {LOCAL_FILE}")
    df = pd.read_csv(LOCAL_FILE, header=None, names=COLUMNS)
    return df
df = dataset()

#Task 1
print("\nFirst five samples:\n", df.head())
print("\nDataset dimensions:", df.shape)
print("\nMissing values per column:\n", df.isnull().sum())
print("\nDescriptive statistics:\n", df.describe())

#Task 2
df.drop(columns="Class").hist(bins=30, figsize=(10, 8))
plt.suptitle("Feature Histograms")
plt.tight_layout()
plt.savefig("histograms.eps", dpi=600)

plt.close()
plt.figure(figsize=(6, 5))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")

plt.tight_layout()
plt.savefig("correlation_heatmap.eps", dpi=600)
plt.close()
plt.figure()
sns.scatterplot(data=df, x="Variance", y="Skewness", hue="Class", palette="Set1")
plt.title("Scatter Plot: Variance vs Skewness by Class")
plt.tight_layout()

plt.savefig("scatter_plot.eps", dpi=600)
plt.close()
plt.figure(figsize=(10, 6))
df.drop(columns="Class").boxplot()
plt.title("Boxplots of Features")
plt.tight_layout()
plt.savefig("boxplots.eps", dpi=600)
plt.close()
print("EDA plots saved: histograms.eps, correlation_heatmap.eps, "
      "scatter_plot.eps, boxplots.eps")

#Task 3
X = df.drop(columns="Class").values
y = df["Class"].values

scaler = StandardScaler()
scaled_x = scaler.fit_transform(X)

training_x, testing_x, training_y, testing_y = train_test_split(
    scaled_x  , y, test_size=0.20, random_state=42, stratify=y
)

print("Training set size:", training_x.shape)
print("Testing set size:", testing_x.shape)


class Perceptron:
    def __init__(self, features, lr =0.01, epochs=50):
        self.eta = lr
        self.epochs = epochs
        self.weights = np.zeros(features)
        self.bias = 0.0
        self.errorsPerepoch = []
        self.weight_prev = []
        self.bias_prev = []

    @staticmethod
    def step_activation(z):
        return np.where(z >= 0, 1, 0)


    def net_input(self, x):
        return np.dot(x, self.weights) + self.bias

    def predict(self, X):
        return self.step_activation(self.net_input(X))


    def fit(self, X, y, verbose=True):
        for epoch in range(1, self.epochs + 1):
            misclassified = 0
            for xi, target in zip(X, y):
                z = self.net_input(xi)
                y_hat = self.step_activation(z)
                update = self.eta * (target - y_hat)
                if update != 0.0:

                    self.weights += update * xi
                    self.bias += update
                    misclassified += 1

            self.errorsPerepoch.append(misclassified)

            self.weight_prev.append(self.weights.copy())
            self.bias_prev.append(self.bias)

            if verbose:
                print(f"Epoch {epoch:3d} | Misclassified: {misclassified:4d} | "
                      f"Weights: {np.round(self.weights, 4)} | "
                      f"Bias: {self.bias:.4f}")

            if misclassified == 0:
                break
        return self


#Task 5
LR = 0.01
EPOCHS = 50
model = Perceptron(features=training_x.shape[1], lr=LR,
                    epochs=EPOCHS)
model.fit(training_x, training_y)

plt.figure()
plt.plot(range(1, len(model.errorsPerepoch) + 1), model.errorsPerepoch,
         marker="o")
plt.xlabel("Epoch")
plt.ylabel("Misclassified Samples")
plt.title(f"Training Error vs Epoch (eta = {LR})")

plt.tight_layout()
plt.savefig("training_error_vs_epoch.eps", dpi=600)
plt.close()
weight_hist = np.array(model.weight_prev)
plt.figure()
for i in range(weight_hist.shape[1]):
    plt.plot(weight_hist[:, i], label=f"w{i+1} ({COLUMNS[i]})")
plt.xlabel("Epoch")
plt.ylabel("Weight Value")
plt.title("Weight Evolution")

plt.legend()
plt.tight_layout()
plt.savefig("weight_evolution.eps", dpi=600)
plt.close()
plt.figure()

plt.plot(model.bias_prev, marker="o", color="darkorange")
plt.xlabel("Epoch")
plt.ylabel("Bias Value")

plt.title("Bias Evolution")
plt.tight_layout()
plt.savefig("bias_evolution.eps", dpi=600)
plt.close()
epoch_table = pd.DataFrame({
    "Epoch": range(1, len(model.errorsPerepoch) + 1),
    "Errors": model.errorsPerepoch,
    "Weight_1": weight_hist[:, 0],
    "Weight_2": weight_hist[:, 1],
    "Bias": model.bias_prev,
})
epoch_table.to_csv("epoch_wise_learning.csv", index=False)
print("\nEpoch-wise learning table saved to epoch_wise_learning.csv")
print(epoch_table.head())

#Task 6
pred_y = model.predict(testing_x)
acc = accuracy_score(testing_y, pred_y)

prec = precision_score(testing_y, pred_y)
recall = recall_score(testing_y, pred_y)
f1 = f1_score(testing_y, pred_y)
cm = confusion_matrix(testing_y, pred_y)
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("Confusion Matrix:\n", cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["Authentic (0)", "Forged (1)"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=600)
plt.close()

summary = pd.DataFrame({
    "Metric": ["Dataset Size", "Train/Test Split", "Learning Rate", "Epochs Run",
               "Final Weights", "Final Bias", "Accuracy", "Precision",
               "Recall", "F1-score"],
    "Value": [df.shape[0], "80/20", LR, len(model.errorsPerepoch),
              np.round(model.weights, 4).tolist(), round(model.bias, 4),
              round(acc, 4), round(prec, 4),
              round(recall, 4), round(f1, 4)]
})
summary.to_csv("training_summary.csv", index=False)
print("\nTraining summary saved to training_summary.csv")

print("\nLearning Rate Comparison")
lrs = [0.001, 0.01, 0.1]
styles = [
    dict(lw=8, alpha=0.35, linestyle="-",  color="tab:blue"),
    dict(lw=5, alpha=0.55, linestyle="--", color="tab:orange"),
    dict(lw=2, alpha=1.0,  linestyle="-",  color="tab:green"),
]
plt.figure()
for lr, style in zip(lrs, styles):
    m = Perceptron(features=training_x.shape[1], lr=lr, epochs=EPOCHS)
    m.fit(training_x, training_y, verbose=False)

    plt.plot(range(1, len(m.errorsPerepoch) + 1), m.errorsPerepoch,
              marker="o", markersize=3, label=f"eta = {lr}", **style)

    print(f"eta = {lr:<6} -> Epochs to converge: {len(m.errorsPerepoch):3d}, "
          f"Final Weights: {np.round(m.weights, 4)}, Final Bias: {m.bias:.4f}")

plt.xlabel("Epoch")
plt.ylabel("Misclassified Samples")
plt.title("Learning Rate Comparison\n(curves overlap — eta only rescales weights, not the error trajectory)")
plt.legend()
plt.tight_layout()
plt.savefig("learning_rate_comparison.eps", dpi=600)
plt.close()

plt.xlabel("Epoch")
plt.ylabel("Misclassified Samples")
plt.title("Learning Rate Comparison")

plt.legend()
plt.tight_layout()
plt.savefig("learning_rate_comparison.eps", dpi=600)
plt.close()
print("\nOptional")
X2 = scaled_x[:, :2]
training_x2, testing_x2, training_y2, testing_y2 = train_test_split(
    X2, y, test_size=0.20, random_state=42, stratify=y
)

model_2d = Perceptron(features=2, lr=0.01, epochs=50)
model_2d.fit(training_x2, training_y2, verbose=False)

x_min, x_max = X2[:, 0].min() - 1, X2[:, 0].max() + 1
y_min, y_max = X2[:, 1].min() - 1, X2[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                      np.linspace(y_min, y_max, 300))
grid_preds = model_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure()
plt.contourf(xx, yy, grid_preds, alpha=0.3, cmap="coolwarm")
sns.scatterplot(x=X2[:, 0], y=X2[:, 1], hue=y, palette="Set1", edgecolor="k")


plt.xlabel("Variance (scaled)")
plt.ylabel("Skewness (scaled)")
plt.title("Decision Boundary (Variance vs Skewness)")
plt.tight_layout()
plt.savefig("decision_boundary.eps", dpi=600)
plt.close()

print("\nComparison with Scikit-learn's Perceptron")
sk_model = SklearnPerceptron(max_iter=EPOCHS, eta0=LR, random_state=42)
sk_model.fit(training_x, training_y)
sk_prediction = sk_model.predict(testing_x)

print("Scratch Perceptron  -> Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1: {:.4f}".format(
    acc, prec, recall, f1))
print("Sklearn Perceptron  -> Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1: {:.4f}".format(
    accuracy_score(testing_y, sk_prediction), precision_score(testing_y, sk_prediction), recall_score(testing_y, sk_prediction), f1_score(testing_y, sk_prediction)))


Dataset downloaded from UCI repository.

First five samples:
    Variance  Skewness  Curtosis  Entropy  Class
0   3.62160    8.6661   -2.8073 -0.44699      0
1   4.54590    8.1674   -2.4586 -1.46210      0
2   3.86600   -2.6383    1.9242  0.10645      0
3   3.45660    9.5228   -4.0112 -3.59440      0
4   0.32924   -4.4552    4.5718 -0.98880      0

Dataset dimensions: (1372, 5)

Missing values per column:
 Variance    0
Skewness    0
Curtosis    0
Entropy     0
Class       0
dtype: int64

Descriptive statistics:
           Variance     Skewness     Curtosis      Entropy        Class
count  1372.000000  1372.000000  1372.000000  1372.000000  1372.000000
mean      0.433735     1.922353     1.397627    -1.191657     0.444606
std       2.842763     5.869047     4.310030     2.101013     0.497103
min      -7.042100   -13.773100    -5.286100    -8.548200     0.000000
25%      -1.773000    -1.708200    -1.574975    -2.413450     0.000000
50%       0.496180     2.319650     0.616630    -0.5866

EDA plots saved: histograms.eps, correlation_heatmap.eps, scatter_plot.eps, boxplots.eps
Training set size: (1097, 4)
Testing set size: (275, 4)
Epoch   1 | Misclassified:   59 | Weights: [-0.068  -0.0938 -0.0678 -0.011 ] | Bias: -0.0300
Epoch   2 | Misclassified:   25 | Weights: [-0.0775 -0.1042 -0.0811  0.0035] | Bias: -0.0400
Epoch   3 | Misclassified:   20 | Weights: [-0.0802 -0.1127 -0.0917 -0.0036] | Bias: -0.0400
Epoch   4 | Misclassified:   19 | Weights: [-0.0883 -0.1196 -0.1001  0.0089] | Bias: -0.0500
Epoch   5 | Misclassified:   20 | Weights: [-0.0991 -0.1295 -0.1037  0.0102] | Bias: -0.0500
Epoch   6 | Misclassified:   17 | Weights: [-0.1027 -0.1277 -0.1134 -0.001 ] | Bias: -0.0600
Epoch   7 | Misclassified:   24 | Weights: [-0.1081 -0.1393 -0.1108 -0.0094] | Bias: -0.0600
Epoch   8 | Misclassified:   22 | Weights: [-0.117  -0.1357 -0.1319 -0.0025] | Bias: -0.0600
Epoch   9 | Misclassified:   20 | Weights: [-0.1176 -0.147  -0.1364 -0.0062] | Bias: -0.0600
Epoch  10 | Miscla


Epoch-wise learning table saved to epoch_wise_learning.csv
   Epoch  Errors  Weight_1  Weight_2  Bias
0      1      59 -0.068025 -0.093809 -0.03
1      2      25 -0.077536 -0.104214 -0.04
2      3      20 -0.080213 -0.112661 -0.04
3      4      19 -0.088281 -0.119595 -0.05
4      5      20 -0.099089 -0.129506 -0.05
Accuracy : 0.9855
Precision: 0.9683
Recall   : 1.0000
F1-score : 0.9839
Confusion Matrix:
 [[149   4]
 [  0 122]]

Training summary saved to training_summary.csv

Learning Rate Comparison
eta = 0.001  -> Epochs to converge:  50, Final Weights: [-0.02 -0.02 -0.02 -0.  ], Final Bias: -0.0100
eta = 0.01   -> Epochs to converge:  50, Final Weights: [-0.19 -0.24 -0.21 -0.02], Final Bias: -0.1000
eta = 0.1    -> Epochs to converge:  50, Final Weights: [-1.94 -2.36 -2.11 -0.19], Final Bias: -1.0000


/tmp/ipykernel_684/1501236726.py:247: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()



Optional



Comparison with Scikit-learn's Perceptron
Scratch Perceptron  -> Accuracy: 0.9855, Precision: 0.9683, Recall: 1.0000, F1: 0.9839
Sklearn Perceptron  -> Accuracy: 0.9745, Precision: 0.9457, Recall: 1.0000, F1: 0.9721


In [2]:
#Additional Tasks
import numpy as np
import matplotlib.pyplot as plt
import os

np.random.seed(0)


def step(z):
    return 1 if z >= 0 else 0


def plot_boundary_2d(w, X, y, title, save_path):
    fig, ax = plt.subplots(figsize=(5, 5))

    colors = ['red' if label == 0 else 'blue' for label in y]
    ax.scatter(X[:, 0], X[:, 1], c=colors, s=180, edgecolors='k', zorder=3)
    for (x1, x2), label in zip(X, y):
        ax.annotate(f"({x1:.0f},{x2:.0f})->{label}", (x1, x2),
                    textcoords="offset points", xytext=(10, 8), fontsize=9)

    xs = np.linspace(-0.5, 1.5, 200)
    w0, w1, w2 = w
    if abs(w2) > 1e-8:
        ys = -(w0 + w1 * xs) / w2
        ax.plot(xs, ys, 'g-', linewidth=2, label='decision boundary')
        Y_grid, X_grid = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
        Z = w0 + w1 * X_grid + w2 * Y_grid
        ax.contourf(X_grid, Y_grid, Z, levels=[-100, 0, 100], colors=['#ffe0e0', '#e0e8ff'], alpha=0.5)
    elif abs(w1) > 1e-8:
        x_line = -w0 / w1
        ax.axvline(x_line, color='g', linewidth=2, label='decision boundary')
    else:
        ax.set_title(title + "\n(no boundary yet: w1=w2=0)")

    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_title(title)
    ax.legend(loc='lower left', fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close(fig)


def plot_boundary_1d(w, X, y, title, save_path):
    fig, ax = plt.subplots(figsize=(5, 2.5))
    colors = ['red' if label == 0 else 'blue' for label in y]
    ax.scatter(X[:, 0], np.zeros_like(X[:, 0]), c=colors, s=220, edgecolors='k', zorder=3)
    for x1, label in zip(X[:, 0], y):
        ax.annotate(f"x={x1:.0f}->{label}", (x1, 0), textcoords="offset points",
                    xytext=(0, 15), ha='center', fontsize=9)

    w0, w1 = w
    if abs(w1) > 1e-8:
        x_thresh = -w0 / w1
        ax.axvline(x_thresh, color='g', linewidth=2, label=f'threshold x={x_thresh:.2f}')
    ax.set_xlim(-1, 2)
    ax.set_yticks([])
    ax.set_xlabel("x")
    ax.set_title(title)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close(fig)


def train_gate(gate_name, X_raw, y, lr=1.0, max_epochs=20, out_dir="plots"):
    n_samples, n_features = X_raw.shape
    X = np.hstack([np.ones((n_samples, 1)), X_raw])
    w = np.zeros(n_features + 1)

    os.makedirs(out_dir, exist_ok=True)
    log_lines = []
    log_lines.append(f"===== {gate_name} GATE =====")
    log_lines.append(f"Initial weights (w0=bias, w1, w2, ...): {w.tolist()}")

    update_count = 0
    if n_features == 2:
        plot_boundary_2d(w, X_raw, y, f"{gate_name}: initial (update 0)\nw={np.round(w,2)}",
                          os.path.join(out_dir, f"{gate_name}_update_000_init.png"))
    else:
        plot_boundary_1d(w, X_raw, y, f"{gate_name}: initial (update 0)\nw={np.round(w,2)}",
                          os.path.join(out_dir, f"{gate_name}_update_000_init.png"))

    converged = False
    for epoch in range(1, max_epochs + 1):
        epoch_updates = 0
        for i in range(n_samples):
            xi = X[i]
            target = y[i]
            net = np.dot(w, xi)
            pred = step(net)
            error = target - pred
            if error != 0:
                w = w + lr * error * xi
                update_count += 1
                epoch_updates += 1
                log_lines.append(
                    f"Epoch {epoch}, sample {X_raw[i]} target={target} pred={pred} "
                    f"error={error} -> weights updated to {np.round(w, 3).tolist()} "
                    f"(update #{update_count})"
                )
                title = f"{gate_name}: after update #{update_count} (epoch {epoch})\nw={np.round(w,2)}"
                fname = os.path.join(out_dir, f"{gate_name}_update_{update_count:03d}.png")
                if n_features == 2:
                    plot_boundary_2d(w, X_raw, y, title, fname)
                else:
                    plot_boundary_1d(w, X_raw, y, title, fname)
        if epoch_updates == 0:
            log_lines.append(f"Epoch {epoch}: no weight changes -> CONVERGED.")
            converged = True
            break

    log_lines.append(f"Final weights: {np.round(w, 3).tolist()}")
    log_lines.append(f"Total updates: {update_count}, Converged: {converged}\n")

    return w, log_lines, update_count


if __name__ == "__main__":
    all_logs = []

    #OR
    X_or = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    y_or = np.array([0, 1, 1, 1])
    w_or, logs_or, n_or = train_gate("OR", X_or, y_or, lr=1.0, out_dir="plots/OR")
    all_logs += logs_or

    #AND
    X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    y_and = np.array([0, 0, 0, 1])
    w_and, logs_and, n_and = train_gate("AND", X_and, y_and, lr=1.0, out_dir="plots/AND")
    all_logs += logs_and

    #NOT
    X_not = np.array([[0], [1]])
    y_not = np.array([1, 0])
    w_not, logs_not, n_not = train_gate("NOT", X_not, y_not, lr=1.0, out_dir="plots/NOT")
    all_logs += logs_not

    with open("perceptron_and_or_not_log.txt", "w") as f:
        f.write("\n".join(all_logs))

    for line in all_logs:
        print(line)

    print("\nFinal weights:")
    print("OR :", np.round(w_or, 3))
    print("AND:", np.round(w_and, 3))
    print("NOT:", np.round(w_not, 3))

/tmp/ipykernel_2175/3569142270.py:41: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc='lower left', fontsize=8)
/tmp/ipykernel_2175/3569142270.py:64: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc='upper right', fontsize=8)


===== OR GATE =====
Initial weights (w0=bias, w1, w2, ...): [0.0, 0.0, 0.0]
Epoch 1, sample [0 0] target=0 pred=1 error=-1 -> weights updated to [-1.0, 0.0, 0.0] (update #1)
Epoch 1, sample [0 1] target=1 pred=0 error=1 -> weights updated to [0.0, 0.0, 1.0] (update #2)
Epoch 2, sample [0 0] target=0 pred=1 error=-1 -> weights updated to [-1.0, 0.0, 1.0] (update #3)
Epoch 2, sample [1 0] target=1 pred=0 error=1 -> weights updated to [0.0, 1.0, 1.0] (update #4)
Epoch 3, sample [0 0] target=0 pred=1 error=-1 -> weights updated to [-1.0, 1.0, 1.0] (update #5)
Epoch 4: no weight changes -> CONVERGED.
Final weights: [-1.0, 1.0, 1.0]
Total updates: 5, Converged: True

===== AND GATE =====
Initial weights (w0=bias, w1, w2, ...): [0.0, 0.0, 0.0]
Epoch 1, sample [0 0] target=0 pred=1 error=-1 -> weights updated to [-1.0, 0.0, 0.0] (update #1)
Epoch 1, sample [1 1] target=1 pred=0 error=1 -> weights updated to [0.0, 1.0, 1.0] (update #2)
Epoch 2, sample [0 0] target=0 pred=1 error=-1 -> weights u